In [20]:
2+2

4

In [21]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
# import seaborn as sns
# import matplotlib.pyplot as plt

import ast
import re
import shutil


In [22]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [23]:
result = pd.read_parquet("./predictions.parquet")
print(result.shape)
result.sample(2)

(216, 11)


,id,template,placeholders,session,cam,constructed_filename,label,top1_pred,correct_top1,top5_preds,correct_top5
163,260930-143609,Putting [something] onto [something],"books, plate",books_2,cam1,260930-143609_cam1.avi,112,48,False,"[112, 12, 106, 144, 48]",True
164,260930-143609,Putting [something] onto [something],"books, plate",books_2,cam2,260930-143609_cam2.avi,112,102,False,"[144, 118, 48, 112, 102]",True


In [24]:
# rename top1_label to top1_pred
result = result.rename(columns={'id':'video_id'})

del result['constructed_filename']

result.sample(3)

,video_id,template,placeholders,session,cam,label,top1_pred,correct_top1,top5_preds,correct_top5
215,260930-150433,Piling [something] up,magazines,piling_1,cam2,48,48,True,"[114, 16, 102, 144, 48]",True
157,260930-143541,Moving [something] up,books,books_2,cam1,45,29,False,"[45, 25, 27, 20, 29]",True
0,260930-122254,Putting [something] next to [something],"electrical extension, plug",plug_2,cam0,107,107,True,"[49, 19, 100, 73, 107]",True


In [25]:
d = np.load("./predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(216, 174)


NpzFile './predictions.npz' with keys: logits, probs, video_ids, labels

# main result

In [30]:

n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()

print(f"Number of samples: {n}")
print(f"Top-1 accuracy: {top1_acc:.4f}")
print(f"Top-5 accuracy: {top5_acc:.4f}")

Number of samples: 216
Top-1 accuracy: 0.4491
Top-5 accuracy: 0.7593


- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [32]:
result.groupby("cam")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

,top1_acc
cam,
cam0,0.41667
cam1,0.51389
cam2,0.41667


In [34]:
result.groupby(['cam']).agg(
    # count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,top1_acc,top5_acc
cam,,
cam0,0.41667,0.73611
cam1,0.51389,0.86111
cam2,0.41667,0.68056


In [35]:
result.groupby(['template']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
template,,,
Attaching [something] to [something],9,0.22222,0.44444
Moving [something] up,36,0.38889,0.75000
Piling [something] up,9,0.22222,0.66667
Plugging [something] into [something],9,0.88889,0.88889
Pushing [something] with [something],36,0.69444,0.97222
Putting [something] next to [something],36,0.41667,0.77778
Putting [something] on a surface,36,0.55556,0.80556
Putting [something] onto [something],36,0.19444,0.61111
Showing a photo of [something] to the camera,9,0.44444,0.55556


In [39]:
result[result.cam=='cam1'].groupby(['template']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
template,,,
Attaching [something] to [something],3,0.33333,0.66667
Moving [something] up,12,0.50000,1.00000
Piling [something] up,3,0.00000,0.66667
Plugging [something] into [something],3,1.00000,1.00000
Pushing [something] with [something],12,0.75000,1.00000
Putting [something] next to [something],12,0.75000,1.00000
Putting [something] on a surface,12,0.50000,0.75000
Putting [something] onto [something],12,0.08333,0.58333
Showing a photo of [something] to the camera,3,0.66667,1.00000


In [38]:
# result[result.cam=='cam1'].groupby(['template','session']).agg(
#     count=('video_id', 'count'),
#     top1_acc=('correct_top1', 'mean'),
#     top5_acc=('correct_top5', 'mean')
# )

# Granular analysis

##### id2label

In [40]:
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}


print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [41]:
result.head(1)

,video_id,template,placeholders,session,cam,label,top1_pred,correct_top1,top5_preds,correct_top5
0,260930-122254,Putting [something] next to [something],"electrical extension, plug",plug_2,cam0,107,107,True,"[49, 19, 100, 73, 107]",True


In [42]:
df = result[['video_id',	'cam'	,'session'	,'placeholders','template',	'label'	,'top1_pred','correct_top1','correct_top5']]
print(df.shape)
df.head(2)

(216, 9)


,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5
0,260930-122254,cam0,plug_2,"electrical extension, plug",Putting [something] next to [something],107,107,True,True
1,260930-122254,cam1,plug_2,"electrical extension, plug",Putting [something] next to [something],107,107,True,True


In [43]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5,prob_0,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260930-122254,cam0,plug_2,"electrical extension, plug",Putting [something] next to [something],107,107,True,True,0.000008,...,0.001373,0.000001,0.000002,0.000003,0.000000,0.000000,0.000044,0.000010,0.000014,0.000002
1,260930-122254,cam1,plug_2,"electrical extension, plug",Putting [something] next to [something],107,107,True,True,0.000000,...,0.000002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000005,0.000000,0.000000


#### Main Analysis 


In [45]:
pd.set_option('display.float_format',  '{:,.4f}'.format)

In [46]:
def avg_prob_in_diff_cases(df: pd.DataFrame, session_type: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[(df['cam'].isin(cams)) & (df['session'].str.contains(session_type))].copy()
    
    r = df_temp.groupby('session').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        # top5acc=('correct_top5', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session').reset_index()

    print("=" * 25, session_type, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos', "=" * 20)

    dist_df = df_temp

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = dist_df['video_id'].dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

In [47]:
df.session.unique()

array(['plug_2', 'lady_2', 'magnet_2', 'books_2', 'plugging_1',
       'showing_photo_of_1', 'attaching_1', 'piling_1'], dtype=object)

In [48]:
avg_prob_in_diff_cases(df, session_type='plug_2', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plug_2 =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
  session  videos  prob_mean  prob_std  top1acc                                     examples
0  plug_2      15     0.0415    0.0721   0.4667  260930-122254, 260930-122317, 260930-122340
==================== Videos ====================
        video_id   cam session                placeholders                                 template label top1_pred correct_top1 correct_top5 prob_107 prob_73 prob_11 prob_19 prob_42
1  260930-122254  cam1  plug_2  electrical extension, plug  Putting [something] next to [something]   107       107         True         True   0.9830  0.0062  0.0052  0.0018  0.0009

Top 5 Probability Breakdown:
  prob_107 ->  0.98298  |  Putting [something] next to [something]
  prob_73  ->  0.00617  |  Pretending to put [something] next to [something]
  prob_11  ->  0.00520  |  Dropping [something] next to [something]
  prob_19  ->  0.00181  |  Hol